# The Grid Module
The grid module provides an interface for library and user-defined modules to add grid points to the NLP.  Despite its name, the grid module does not create its own grid points.  Its primary role is to partition the grid into phases and subphases.  It also provides a mechanism for built-in and custom modules to add their grid points to the NLP.  The model definition of the grid module is presented at the end of this notebook.

**NOTE:** In Amplify, nearly every module depends on the grid module.

In [ ]:
from amplpy import AMPL
ampl = AMPL()

In [ ]:
%%ampl_eval
#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#* grid
#----------- modules
# model-defined modules;
set MOD_MODULES;
set MOD_GRID {MOD_MODULES}; # this set will be redeclared later

# data-defined modules
set DAT_MODULES default {};
set DAT_GRID {DAT_MODULES};

# all modules
set MODULES = MOD_MODULES union DAT_MODULES;
set GRID_MODULES {i1 in MODULES} := 
  if i1 in MOD_MODULES then MOD_GRID[i1]  
  else DAT_GRID[i1];

#----------- phases
# dynamics are constant within a phase
set SUBPHASES within Reals;
set SUBPHASES_MBRS = {'PHASE', 'BEG', 'END'};
param subphases {SUBPHASES, SUBPHASES_MBRS};

# subphases must map to phase through floor function
check {i1 in SUBPHASES} : subphases[i1, 'PHASE'] = floor(i1);
# beginning must be less than end
check {i1 in SUBPHASES} : subphases[i1, 'BEG'] <= subphases[i1, 'END'];
# make sure sets don't overlap
check {i1 in SUBPHASES, i2 in SUBPHASES : i1 != i2} : inter {i3 in {i1, i2}} union {i4 in {'BEG', 'END'}} {subphases[i3, i4]} within {};

set PHASES = setof {i1 in SUBPHASES} subphases[i1, 'PHASE'];

#----------- grid points
param GRID_MAX >= 0;

set GRID_MBRS = {0} union SUBPHASES union PHASES;

set GRID {i1 in GRID_MBRS} ordered by [0, GRID_MAX] :=
  if i1 = 0 then # add grid points from registered modules
    union {i2 in MODULES} GRID_MODULES[i2]
  else if i1 in Integers then # collect grid points for each phase
    union {i2 in SUBPHASES : floor(i2) = i1} GRID[i2]
  else {i2 in GRID[0] : subphases[i1, 'BEG'] <= i2 <= subphases[i1, 'END']};  # collect grid points for each subphase

#----------- decision variable: time
var t {GRID[0]}; # time
#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#* grid!

In [ ]:
#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#* grid
# model has been processed, so now add data specific rules for modules in the model file
redeclare set MOD_GRID {i1 in MOD_MODULES} := 
  if i1 = 'ODE' then ODE_GRID
  else if i1 = 'PIM' then union {(i2,i3) in PIM} {i2, i3}
  else if i1 = 'BEZ' then union {i2 in BEZ_LBLS, i3 in BEZ_SUBGRID[i2]} bez_subgrid[i2, i3, 'BEG']..bez_subgrid[i2, i3, 'END']
  else {};
#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#* grid!

# Grid Data Section

## About the Example
In the example below, 26 grid point will be created from 0 to 25.  There is one subphase with label `1.1`.  It has the same equations of motion as other subphases in phase 1.  The subphase is defined across grid points 0 to 25, i.e., the entire grid.

In [ ]:
%%ampl_eval
data;
#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#* grid
# params
param GRID_MAX := 25.;
set SUBPHASES := 1.1;
param subphases := [*,:]
     :PHASE   BEG   END :=
1.1   1.       0.    25.;
#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#* grid!

#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#* bezier
set BEZ_LBLS := 0 1;

set BEZ_SUBGRID[0] := 1;
set BEZ_SUBGRID[1] := 1;

param bez_subgrid := [*,*,:]
     :BEG   END :=
0 1   0.     25.
1 1   0.     25.;
#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#* bezier!

#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#* eom
set PIM := (0, 25);
#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#* eom!

#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#* Runge-Kutta
set ODE_SOLVERS := HERM_SIMP;
param ODE_K := [*]
HERM_SIMP   3.;

set ODE_SUBGRID[HERM_SIMP] := (0, 25);
#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#* Runge-Kutta!

#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#* grid
# activate model-define modules in the model file
set MOD_MODULES := ODE PIM BEZ;
#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#* grid!

model;